# 3. De novo design: RFdiffusion → ProteinMPNN
**Question:** Can we propose a protein without starting from a natural sequence or structure?

RFdiffusion generates one 60-residue backbone from noise. ProteinMPNN samples three sequences for it, and ESMFold checks whether one sequence predicts a similar fold. **Target: under 10 minutes after setup on a free Colab GPU; timing needs a classroom trial.**


## Before class
Choose **Runtime → Change runtime type → T4 GPU**, with **runtime version 2026.07** (already selected in this notebook). Run the two setup cells once and keep the runtime connected. The shared tools can be reused later in the class. ESMFold needs a large download; setup time is separate from this demo. Start the timed walkthrough at step 1.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_MPNN()
load_RFdiffusion()
load_ESMFold()


## 1. Generate a backbone
We specify only length. RFdiffusion removes noise over 50 steps to propose a backbone. Rotate the result and inspect its helices, sheets, and loops.


In [ ]:
reference_pdb = RFdiffusion(length=60, steps=50, seed=7)
view_structure(reference_pdb)


## 2. Give the backbone a sequence
RFdiffusion's placeholder glycine sequence is not a designed protein. ProteinMPNN supplies the amino acid sequences.


In [ ]:
designs = MPNNdesign(reference_pdb, num_sequences=3, temperature=0.1, seed=7)


## 3. Compare the designs
There is no wild-type sequence here. Align the three designs to each other; identity is reported relative to design 1. Which residues agree despite different sampling?


In [ ]:
alignment(designs)


## 4. Does the sequence predict the intended fold?
Fold just one sequence to keep the demo short. ESMFold predicts from sequence alone; it does not receive our reference coordinates. One recycle is a fast check. RMSD measures agreement with the reference; pLDDT measures the predictor's local confidence. Neither demonstrates experimental folding or function.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Inspect
- Does the predicted fold match RFdiffusion's proposed backbone? Identify one agreement and one discrepancy.
- Why do we need both a sequence designer and an independent structure prediction step after backbone generation?

[RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) · [Colab adaptation](https://github.com/sokrypton/RFdiffusion) · [ProteinMPNN](https://github.com/dauparas/ProteinMPNN) · [ESMFold](https://github.com/facebookresearch/esm)


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))
